In [1]:
import requests

import pandas as pd

from google.cloud import bigquery

In [2]:
KEY = "579804b0fcedbcee84836facd3ac8db7"

In [ ]:
from pathlib import Path

from google.oauth2 import service_account

BQ_PROJECT_ID = "gedas-analytics-dev"
BQ_DATASET_ID = "external__google_sheets"
BQ_TABLE_ID = "stocks_cost_basis"
CREDENTIALS_PATH = Path(r"d:/Coding/DAE/Airflow/credentials/gc-gedas-analytics-dev-sa.json")

credentials = service_account.Credentials.from_service_account_file(
    CREDENTIALS_PATH,
    scopes=["https://www.googleapis.com/auth/bigquery"],
)
client = bigquery.Client(project=BQ_PROJECT_ID, credentials=credentials)
symbols_df = client.query(
    f"""
    SELECT DISTINCT ticker AS symbol
    FROM `{BQ_PROJECT_ID}.{BQ_DATASET_ID}.{BQ_TABLE_ID}`
    WHERE ticker IS NOT NULL
    """
).to_dataframe()
symbols = symbols_df["symbol"].tolist()
symbols

In [4]:
response = requests.get(
    "https://api.marketstack.com/v2/eod/latest",
    params={
        "access_key": KEY,
        "symbols": ",".join(symbols),
        "limit": max(len(symbols), 1),
    },
)
payload = response.json()
if payload.get("error"):
    raise RuntimeError(payload["error"])

df = pd.DataFrame(payload["data"])[
    ["date", "name", "symbol", "close", "price_currency", "exchange_code"]
]
df

RuntimeError: {'code': 'usage_limit_reached', 'message': 'Your monthly usage limit has been reached. Please upgrade your Subscription Plan.'}